# Load in data

In [12]:
import pandas as pd

def pretty_sample(df, col, n=20, random_state=42):
    print("Sampling from column:", col)
    samp_df = df[col].sample(n=n, random_state=random_state)
    for i, val in enumerate(samp_df):
        print(f"{i+1}. {val}")
        print("---")


n = pd.read_csv("../data/clean/init_normbank_predictions_3000.csv")
# Index(['setting', 'behavior', 'setting-behavior', 'constraints',
#        'constraints_given', 'constraint_predict', 'norm', 'label', 'split',
#        'idx'],

s = pd.read_csv("../data/clean/ssl_stimuli_unannot.csv")
# Index(['rot', 'action', 'situation', 'ssl_domain', 'agreement_condition'], dtype='object')

# Normbank

The way normbank is structured is that we have three core fields:
1. setting - where the behavior is taking place
2. behavior - what the person is doing
3. constraints - these are additional details like who the person doing the action is.

We turn this all into a string like "{behavior} at a {setting} where {constraints}".




## Clean normbank

We have already filtered to roles where the constraint is simple, but there is an awkward placeholder that says [PERSON] so there are different ways to handle this.

Let's also inspect the different norm labels: taboo, normal, expected.

In [5]:
norm_label_map = {
    0: "taboo",
    1: "normal",
    2: "expected"
}
n['norm_label'] = n['label'].map(norm_label_map)

In [13]:
def turn_into_row(x, replace_person=True):
    s = f"{x['behavior']} at a {x['setting']} where {x['constraints']}"
    if replace_person:
        s = s.replace("[PERSON]'s", "the person's")
        s = s.replace("[OTHER]'s", "the other person's")
    else:
        pass
    return s


norm_label_map = {
    0: "taboo",
    1: "normal",
    2: "expected"
}
n['norm_label'] = n['label'].map(norm_label_map)

n['stimulus_replace'] = n.apply(turn_into_row, axis=1)
n['stimulus_noreplace'] = n.apply(lambda x: turn_into_row(x, replace_person=False), axis=1)


print("RANDOM SAMPLES")
pretty_sample(n, 'stimulus_noreplace')
print("====================")
pretty_sample(n, 'stimulus_replace')


RANDOM SAMPLES
Sampling from column: stimulus_noreplace
1. use a hair dryer at a barber shop or beauty salon where [PERSON]'s role is 'nail technician'
---
2. play games on your phone at a conference room where meeting-related time is 'during the meeting'
---
3. leave the register unattended at a cafe where [PERSON]'s role is 'dishwasher'
---
4. make a loud noise at a auditorium where [PERSON]'s behavior is 'alerting to a fire'
---
5. play a contact sport at a athletic field where [PERSON]'s role is 'coach'
---
6. talk about money at a taxi where [PERSON]'s age bracket is 'toddler'
---
7. argue with someone at a restaurant where [PERSON]'s role is 'manager'
---
8. buy a souvenir at a amusement park where [PERSON]'s role is 'ride operator'
---
9. racing at a racetrack where [PERSON]'s role is 'exercise rider'
---
10. attempt to summit at a mountain where [PERSON]'s role is 'rock climber'
---
11. shave your face at a locker room where [PERSON]'s behavior is 'block the only mirror'
---
12

## Explore by norm label

Normbank labels actions by ['taboo', 'normal', 'expected'], so let's sample some examples from each.

In [14]:
print("SAMPLE BY NORM LABEL")
for label in ['taboo', 'normal', 'expected']:
    print(f"START <--- NORM LABEL == '{label.upper()}' --->")
    pretty_sample(n[n['norm_label'] == label], 'stimulus_replace')
    print(f"END <--- NORM LABEL == '{label.upper()}' --->")
    print("====================")
    print("\n"*3)


SAMPLE BY NORM LABEL
START <--- NORM LABEL == 'TABOO' --->
Sampling from column: stimulus_replace
1. curse at a doctors office where the person's behavior is 'arguing with someone'
---
2. run in the aisle at a auditorium where the person's role is 'concession stand worker'
---
3. flirt with someone at a fair where the person's marriage is not 'single'
---
4. conduct individual and group therapy sessions at a alcoholics ward where the person's role is 'doctor'
---
5. follow customers around the store at a camera shop where the person's role is 'store director'
---
6. lick a petri dish at a biology laboratory where location licked is 'inside'
---
7. talk about religion at a circus where the person's behavior is 'avoid eye contact with someone'
---
8. race someone at a pool where the person's role is 'lifeguard'
---
9. wear shoes at a kitchen where the person's behavior is 'using bare hands to handle food'
---
10. flirt with someone at a garage where the person's behavior is 'fight someon

# Social Chemistry

Here's how social chemistry is structured. We have three fields:

1. action - what the person is doing
2. situation - the context in which the action is taking place
3. rot - the high level rule of thumb associated with the action-situation pair

Note: We did have a list of profanity words but I think we need some and can manually filter too. Crowdworkers sort of trolled the authors and a lot of these were explicit before the profanity filter.

There are three ways to present stimuli from this dataset:

1. Rate the appropriateness of the action in a given situation
2. Rate the agreement with the rule of thumb
3. Some kind of hybrid (but I think this would be confusing, so really just two ways)



## Action-Situation

It's originally called ['action', 'context'] but we renamed to ['action', 'situation'] because we thought that was clearer. The action is fairly specific but sometimes the context/situation is super vague (IMO). Maybe it makes to call it context?


In [15]:
def combine_action_situation(row):
    return f"Action: {row['action']}\nContext: {row['situation']}"

s['act_sit'] = s.apply(combine_action_situation, axis=1)
pretty_sample(s, 'act_sit')



Sampling from column: act_sit
1. Action: being gay just because you're in prison.
Context: telling my friend in prison it's ok to be gay for the stay
---
2. Action: expecting to have privacy from a parent that they rely on for everything.
Context: being mad at my mom for checking my phone
---
3. Action: starting relationships in the workplace.
Context: I'm married and I want to bang a hot coworker who is into me
---
4. Action: getting your child a pet.
Context: That morning, Casey cried when her gift was a new puppy!
---
5. Action: Hating babies.
Context: I really, REALLY hate babies
---
6. Action: Your girlfriend getting mad at you when you tell her you have a crush on someone else.
Context: having a crush on a girl and telling my girlfriend
---
7. Action: letting your sibling ride your horse.
Context: I got upset until i find out my brother was riding him.
---
8. Action: helping out your parents if you don't feel like it.
Context: not wanting to help out my dad and his wife
---
9. Ac

## ROT

The other way we can use Social Chemistry is to ask people whether they agree with the rule of thumb (ROT). This can either be just rot alone or in conjunction with
the action-situation.

In [16]:
def combine_rot_action_situation(row):
    return f"ROT: {row['rot']}\nAction: {row['action']}\nContext: {row['situation']}"

s['rot_act_sit'] = s.apply(combine_rot_action_situation, axis=1)


pretty_sample(s, 'rot')
print("\n"*3)
pretty_sample(s, 'rot_act_sit')


Sampling from column: rot
1. It's wrong to be gay just because you're in prison.
---
2. A child shouldn't expect to have privacy from a parent that they rely on for everything.
---
3. It's bad to start relationships in the workplace.
---
4. It's good to get your child a pet.
---
5. It's bad to hate babies.
---
6. Your girlfriend is expected to get mad at you, if you tell her you have a crush on someone else.
---
7. You shouldn't let your sibling ride your horse.
---
8. You don't have to help out your parents if you don't feel like it.
---
9. It is good to change a behavior that a friend does not like.
---
10. You shouldn't be friends with your ex if they still have feelings for you.
---
11. You shouldn't have an affair.
---
12. If you're going to attack someone, you should do it with all your strength.
---
13. You should not sabotage your personal relationships.
---
14. You have a right to think women are inferior if that's what you really believe.
---
15. It's good to make your voice 

## Stratifying by agreement condition

In [17]:
for agreement in ['low_agreement', 'medium_agreement', 'high_agreement']:
    print(f"START <--- AGREEMENT == '{agreement.upper()}' --->")
    pretty_sample(s[s['agreement_condition'] == agreement], 'rot_act_sit', n=9)
    print(f"END <--- AGREEMENT == '{agreement.upper()}' --->")
    print("====================")
    print("\n"*3)

START <--- AGREEMENT == 'LOW_AGREEMENT' --->
Sampling from column: rot_act_sit
1. ROT: It's expected a husband put his foot down when his wife is behaving improperly.
Action: a husband putting his foot down when his wife is behaving improperly.
Context: insisting my wife stop seeing her best friend and their kids
---
2. ROT: You shouldn't tell others about your sexual identity.
Action: telling others about your sexual identity.
Context: not telling my grandpa I am transgender
---
3. ROT: It's good to tell your roommate to get over something they don't like that you did.
Action: telling your roommate to get over something they don't like that you did.
Context: telling my roommate to get over my dishwashing
---
4. ROT: It's wrong to know what death is really like.
Action: knowing what death is really like.
Context: I know what death is really like
---
5. ROT: You shouldn't have your partner over when you have your friend over at your house.
Action: having your partner over when you have 